In [1]:
import numpy as np
import tensorflow as tf
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import TruncatedNormal
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy
import torch
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, TrainingArguments
from transformers import AutoTokenizer, BertForSequenceClassification
from datasets import load_dataset, Dataset

2025-03-07 20:01:57.784588: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-07 20:01:57.808340: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741357917.832591 2235623 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741357917.839577 2235623 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-07 20:01:57.864838: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
df_train = pd.read_csv("Train_title.csv", encoding = 'utf-8')

In [3]:
df_test = pd.read_csv("Test_title.csv", encoding = 'utf-8')

In [4]:
df = pd.concat([df_train, df_test], ignore_index=True)

In [5]:
df.head(10)

,news_category,news_title
0,Tamil Nadu,மாவோயிஸ்ட் தாக்குதலில் வீரமரணம் : துப்பாக்கி க...
1,Tamil Nadu,காஞ்சிபுரத்தில் இன்று அதிகாலை மளிகை கடை குடோனி...
2,Cinema,கிளிப்பிங்ஸ்
3,World,அமெரிக்காவின் டெக்சாஸ் மாகாணம்
4,Sports,விளையாட்டு அமைச்சகம் பரிந்துரை பி.வி.சிந்துவுக...
5,Crime,எழும்பூரில் உள்ள வணிக வளாகத்தில் 14 கடைகளை உடை...
6,Sports,டோனியுடன் உரசல் இல்லை
7,District News,தொன்னாடு ஊராட்சியில் வங்கி தலைவர் தேர்வு
8,India,நடிகை வீட்டில் கள்ள நோட்டுக்கள்
9,Tamil Nadu,ஜனநாயக உரிமையை காலில் போட்டு ஜெயலலிதா அரசு மித...


In [6]:
df.count()

news_category    64034
news_title       64034
dtype: int64

In [7]:
import nltk
from nltk.tokenize import word_tokenize
from transformers import AutoTokenizer
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

titles = df['news_title'].astype(str).tolist()

# Define tokenizers
wordpiece_tokenizer = AutoTokenizer.from_pretrained("bert_wordpiece/")
sentencepiece_tokenizer = AutoTokenizer.from_pretrained("bert_senpiece/")
byte_bpe_tokenizer = AutoTokenizer.from_pretrained("bert_bbpe/")

In [ ]:
import numpy as np
import pandas as pd

def tokenize_and_analyze(tokenizer, tokenizer_name, titles):
    token_counts = []
    subword_fragmentation = []
    all_tokens = []
    oov_counts = []
    total_chars = 0
    total_tokens = 0
    
    for title in titles:
        # Tokenize the title
        tokens = tokenizer.tokenize(title)
        token_counts.append(len(tokens))
        subword_fragmentation.append(len(tokens) / len(title.split()))  # Fragmentation Score
        all_tokens.extend(tokens)
        
        # Calculate OOV tokens
        if hasattr(tokenizer, "convert_tokens_to_ids"):
            oov_count = sum(1 for token in tokens if tokenizer.convert_tokens_to_ids(token) == tokenizer.unk_token_id)
            oov_counts.append(oov_count)
        
        # Calculate total characters and tokens for compression ratio
        total_chars += len(title)
        total_tokens += len(tokens)
    

    
    # Calculate OOV rate
    oov_rate = np.mean(oov_counts) if oov_counts else 0
    
    # Calculate compression ratio (tokens vs characters)
    compression_ratio = total_chars / total_tokens if total_tokens > 0 else 0
    
    # Calculate token overlap (Jaccard similarity)
    token_overlap = []
    for i in range(len(titles) - 1):
        tokens1 = set(tokenizer.tokenize(titles[i]))
        tokens2 = set(tokenizer.tokenize(titles[i + 1]))
        intersection = len(tokens1.intersection(tokens2))
        union = len(tokens1.union(tokens2))
        jaccard = intersection / union if union > 0 else 0
        token_overlap.append(jaccard)
    
    avg_token_overlap = np.mean(token_overlap) if token_overlap else 0
    
    return {
        "avg_token_count": avg_token_count,
        "avg_fragmentation": avg_fragmentation,
        "oov_rate": oov_rate,
        "compression_ratio": compression_ratio,
    }

# Analyze tokenizers
results = {}
results["WordPiece"] = tokenize_and_analyze(wordpiece_tokenizer, "WordPiece", titles)
results["SentencePiece"] = tokenize_and_analyze(sentencepiece_tokenizer, "SentencePiece", titles)
results["Byte BPE"] = tokenize_and_analyze(byte_bpe_tokenizer, "Byte BPE", titles)

# Convert results to DataFrame for comparison
df_results = pd.DataFrame.from_dict(results, orient='index')
print(df_results)